In [ ]:
#packages
from pyspark.sql import SparkSession
import pyspark.sql.dataframe
import pyspark.sql.functions as f
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
import math as m
from pyspark.sql.window import Window
from pyspark.ml.regression import GBTRegressor
from pyspark.ml import Pipeline
from functools import reduce
from pyspark.ml.evaluation import RegressionEvaluator


In [ ]:
#creating a spark session
spark = (
    SparkSession.builder.appName("Project 1")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)

In [ ]:
#remove unnecessary output
spark.sparkContext.setLogLevel("ERROR")
import logging
logging.getLogger('org.apache.spark.sql.execution.window.WindowExec').setLevel(logging.ERROR)

In [ ]:
#Functions
def create_pl(lags, time_length, type):

    """Creates a pipeline for model creation"""

    categorical_cols = ["biz_tags","rev_band"]
    indexer = StringIndexer(inputCols=categorical_cols, outputCols=[f"{c}_idx" for c in categorical_cols], handleInvalid="keep")
    encoder = OneHotEncoder(inputCols=[f"{c}_idx" for c in categorical_cols],
                        outputCols=[f"{c}_vec" for c in categorical_cols])
    
    feature_cols = (
        ["revenue"] +
        [f"rev_lag{l}" for l in lags] +
        [f"rev_mean_{w}" for w in [3,6,12]] +
        [f"rev_std_{w}" for w in [3,6,12]] +
        ["sin_month", "cos_month"] +
        [f"{c}_vec" for c in categorical_cols]
        )

    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
    gbt = GBTRegressor(labelCol=f"{type}_{time_length}m", featuresCol="features", maxIter=100)   
    pipeline = Pipeline(stages=[indexer, encoder, assembler, gbt])
    return pipeline

def find_NULL(dfs):

    """Finds entries with NULL values over multiple datasets"""

    for df in dfs:
        condition = f.lit(False)
        
        condition = condition | f.col('rev_future_6m').isNull() & f.col('rev_future_12m').isNotNull()

        df.filter(condition).show()
    return df.filter(condition).show()

def impute_rev_lags_by_business(data, lags, group_col, order_col='year_month'):
    """
    Imputes missing revenue lag columns (rev_lag1, rev_lag3, etc.)
    with the mean of that lag for each business.
    """
    base_window = Window.partitionBy(group_col).orderBy(order_col) \
                        .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

    for l in lags:
        colname = f"rev_lag{l}"
        first_col = f"{colname}_first"
        
        # compute business-specific mean and fill nulls
        data = (
            data
            .withColumn(first_col, f.first(col(colname), ignorenulls=True).over(base_window))
            .withColumn(
                colname,
                f.when(col(colname).isNull(), col(first_col)).otherwise(col(colname))
            )
            .drop(first_col)
        )

    return data

def create_model(data, time_length, type):
    
    """
    Creates a GBT model for predicting future revenue and growth
    """

    lags=[1,3,6,12]
    required_cols = [f"{type}_{time_length}m"]
    data=data.na.drop(subset=required_cols)
    data=impute_rev_lags_by_business(data, lags, 'merchant_abn', 'year_month')
    data=data.na.fill(0)
    pipeline = create_pl(lags, time_length, type)
    train, test = data.randomSplit([0.8,0.2], seed=42)
    model = pipeline.fit(train)
    predictions = model.transform(test)
    
    return predictions
    
def performance(predictions, time_length, type):
    """
    Checks performance using rmse and mae
    """
    
    rmse_rev = RegressionEvaluator(
        labelCol=f"{type}_{time_length}m",
        predictionCol="prediction",
        metricName="rmse"
    )

    mae_rev = RegressionEvaluator(
        labelCol=f"{type}_{time_length}m",
        predictionCol="prediction",
        metricName="mae"
    )


    rmse = rmse_rev.evaluate(predictions)
    mae = mae_rev.evaluate(predictions)

    
    ranked = (
        predictions.groupBy("merchant_abn")
        .agg(f.avg("prediction").alias(f"pred_{type}"))
        
    )

    window = Window.partitionBy(f.lit(1)).orderBy(f.desc(f"pred_{type}"))
    
    # Add ranking column
    ranked = ranked.withColumn(f"{type}_rank", f.row_number().over(window))
    #print(f"RMSE: {rmse:.4f}")
    return mae, rmse, ranked

def combine(growth, revenue):
    """
    Combines ranking dfs into a composite ranking    
    """
    g=0.2
    rev=0.8
    combined = growth.join(revenue,
        on="merchant_abn",
        how="inner"  # or "outer" if some merchants are missing in one DF
        )

    # Create a composite rank (sum of ranks, lower is better)
    combined = (combined.withColumn(
            "composite_rank",
            f.round(g*f.col("growth_rank") + rev*f.col("rev_future_rank"),4)
            )
        .orderBy('composite_rank', ascending=True)
        )

    # Optionally, sort by the composite rank
    combined = combined.orderBy(f.asc("composite_rank"))
    return combined
    

In [ ]:
#read in merchant_transactions
merchant_transactions=spark.read.parquet('../data/curated/merchant_transactions')

In [ ]:
#group by merchant abn, create joining table
merchant_abn_name=merchant_transactions.groupBy('merchant_abn', 'business').count()
merchant_abn_name

In [ ]:
#write file
merchant_abn_name.write.parquet("../data/curated/merchant_abn_name", mode='overwrite')

In [ ]:
#seperate column for month/year, drop other columns
merchant_transactions=merchant_transactions.withColumn("year_month", f.date_format("order_datetime", "yyyy-MM"))
merchant_transactions=merchant_transactions.drop('user_id', 'business', 'order_datetime')

In [ ]:
merchant_transactions

In [ ]:
#aggregate by merchant and month
month_agg=(merchant_transactions.groupBy('merchant_abn','year_month', 'biz_tags', 'rev_band')
                                .agg(f.round(f.mean('take_rate'),4).alias('ave_take_rate'),
                                     f.round(f.sum('dollar_value'),4).alias('revenue'),
                                     (f.round(col('revenue')*col('ave_take_rate'),4)/100).alias('taking'))
)

In [ ]:
month_agg=month_agg.orderBy('merchant_abn', 'year_month')
month_agg

In [ ]:
#find takings per merchant
taking=month_agg.groupBy('merchant_abn').agg(
                    f.sum('taking').alias('total_taking')
)

window = Window.partitionBy(f.lit(1)).orderBy(f.desc('total_taking'))
taking=taking.withColumn('rank', f.row_number().over(window))
taking

In [ ]:
#write file
taking.write.parquet("../data/curated/current_tot_taking_rank", mode="overwrite")

In [ ]:
#create time windows and lag values
window = Window.partitionBy("merchant_abn").orderBy("year_month")
lags = [3,6,12]
new_data=month_agg.withColumn(f"rev_lag1", f.lag("revenue", 1).over(window))
for lag in lags:
    new_data = new_data.withColumn(f"rev_lag{lag}", f.lag("revenue", lag).over(window))

new_data

In [ ]:
#find mean over time periods
for window_size in [3,6,12]:
    roll_window = Window.partitionBy("merchant_abn").orderBy("year_month").rowsBetween(-window_size+1,0)
    new_data = (
        new_data
        .withColumn(f"rev_mean_{window_size}", f.mean("revenue").over(roll_window))
        .withColumn(f"rev_std_{window_size}", f.stddev("revenue").over(roll_window))
    )

new_data

In [ ]:
#add growth and future revenue over next time period
new_data = (
    new_data
    .withColumn("rev_future_1m", f.lead("revenue", 1).over(window))
    .withColumn("growth_1m", (f.col("rev_future_1m") - f.col("revenue")) / f.col("revenue"))
    .withColumn("rev_future_3m", f.lead("revenue", 3).over(window))
    .withColumn("growth_3m", (f.col("rev_future_3m") - f.col("revenue")) / f.col("revenue"))
    .withColumn("rev_future_6m", f.lead("revenue", 6).over(window))
    .withColumn("growth_6m", (f.col("rev_future_6m") - f.col("revenue")) / f.col("revenue"))
    .withColumn("rev_future_12m", f.lead("revenue", 12).over(window))
    .withColumn("growth_12m", (f.col("rev_future_12m") - f.col("revenue")) / f.col("revenue"))
)

new_data

In [ ]:
#seasonality
new_data = (
    new_data
    .withColumn("month", f.month("year_month"))
    .withColumn("sin_month", f.sin(2 * m.pi * col("month") / 12))
    .withColumn("cos_month", f.cos(2 * m.pi * col("month") / 12))
)

new_data

This section creates models for growth and future revenue over each time period of 1, 3, 6 and 12 months.

In [ ]:
predictions_growth_1 = create_model(new_data, 1, 'growth')

In [ ]:
#errors
mae_g_1, rmse_g_1, ranked_growth_1 = performance(predictions_growth_1, 1, 'growth')
mae_g_1, rmse_g_1

In [ ]:
predictions_rev_1=create_model(new_data, 1, 'rev_future')

In [ ]:
#errors
mae_r_1, rsme_r_1, ranked_rev_1 = performance(predictions_rev_1, 1, 'rev_future')
mae_r_1, rsme_r_1


In [ ]:
#creates composite rank for merchants
composite_rank_1m=combine(ranked_growth_1, ranked_rev_1)
window = Window.partitionBy(f.lit(1)).orderBy(f.asc("composite_rank"))
    
# Add ranking column
composite_rank_1m = composite_rank_1m.withColumn("composite_rank_indx", f.row_number().over(window))
composite_rank_1m

In [ ]:
predictions_rev_3 = create_model(new_data, 3,'rev_future')

In [ ]:
#errors
mae_r_3, rmse_r_3, ranked_rev_3 = performance(predictions_rev_3, 3, 'rev_future')
mae_r_3, rmse_r_3

In [ ]:
predictions_growth_3 = create_model(new_data, 3, 'growth')

In [ ]:
#errors
mae_g_3, rmse_g_3, ranked_growth_3 = performance(predictions_growth_3, 3, 'growth')
mae_g_3, rmse_g_3

In [ ]:
#creates composite rank for merchants
composite_rank_3m=combine(ranked_growth_3, ranked_rev_3)
#window = Window.partitionBy(f.lit(1)).orderBy(f.asc("composite_rank"))
    
# Add ranking column
composite_rank_3m = composite_rank_3m.withColumn("composite_rank_indx", f.row_number().over(window))
composite_rank_3m

In [ ]:
predictions_rev_6 = create_model(new_data, 6, 'rev_future')

In [ ]:
#errors
mae_r_6, rmse_r_6, ranked_rev_6=performance(predictions_rev_6, 6, 'rev_future')
mae_r_6, rmse_r_6

In [ ]:
predictions_growth_6 = create_model(new_data, 6, 'growth')

In [ ]:
#errors
mae_g_6, rmse_g_6, ranked_growth_6 = performance(predictions_growth_6, 6, 'growth')
mae_g_6, rmse_g_6

In [ ]:
composite_rank_6m=combine(ranked_growth_6, ranked_rev_6)
composite_rank_6m = composite_rank_6m.withColumn("composite_rank_indx", f.row_number().over(window))
composite_rank_6m

In [ ]:
predictions_rev_12 = create_model(new_data, 12, 'rev_future')

In [ ]:
#errors
mae_r_12, rmse_r_12, ranked_rev_12 = performance(predictions_rev_12, 12, 'rev_future')
mae_r_12, rmse_r_12

In [ ]:
predictions_growth_12 = create_model(new_data, 12, 'growth')

In [ ]:
#errors
mae_g_12, rmse_g_12, ranked_growth_12 = performance(predictions_growth_12, 12, 'growth')
mae_g_12, rmse_g_12

In [ ]:
composite_rank_12m=combine(ranked_growth_12, ranked_rev_12)
composite_rank_12m = composite_rank_12m.withColumn("composite_rank_indx", f.row_number().over(window))
composite_rank_12m

In [ ]:
#combine all ranks from all windows
rank_1 = composite_rank_1m.withColumnRenamed("composite_rank_indx", "rank_1").drop('pred_growth', 'growth_rank', 'pred_rev_future', 'rev_future_rank', 'composite_rank')
rank_3 = composite_rank_3m.withColumnRenamed("composite_rank_indx", "rank_3").drop('pred_growth', 'growth_rank', 'pred_rev_future', 'rev_future_rank', 'composite_rank')
rank_6 = composite_rank_6m.withColumnRenamed("composite_rank_indx", "rank_6").drop('pred_growth', 'growth_rank', 'pred_rev_future', 'rev_future_rank', 'composite_rank')
rank_12 = composite_rank_12m.withColumnRenamed("composite_rank_indx", "rank_12").drop('pred_growth', 'growth_rank', 'pred_rev_future', 'rev_future_rank', 'composite_rank')

# Join on merchant_abn
combined = reduce(lambda a, b: a.join(b, on="merchant_abn", how="outer"), [rank_1,rank_3,rank_6,rank_12])

combined


In [ ]:
#find weights of data
total = combined.count()
weights_df = combined.select([
    (f.count(col(c)).alias(c)) for c in ["rank_1", "rank_3", "rank_6", "rank_12"]
    ])

weights_row = weights_df.collect()[0].asDict()
weights = {k: v / total for k, v in weights_row.items()}

print(weights)

In [ ]:
#create combined composite ranking
combined = combined.withColumn(
    "composite_rank",
    (
        f.coalesce(col("rank_1"), f.lit(0)) * (weights["rank_1"]) +
        f.coalesce(col("rank_3"), f.lit(0)) * (weights["rank_3"])+
        f.coalesce(col("rank_6"), f.lit(0)) * (weights["rank_6"])+
        f.coalesce(col("rank_12"), f.lit(0))* (weights["rank_12"])
    ) / (
        f.when(col("rank_1").isNotNull(), weights["rank_1"]).otherwise(0) +
        f.when(col("rank_3").isNotNull(), weights["rank_3"]).otherwise(0) +
        f.when(col("rank_6").isNotNull(), weights["rank_6"]).otherwise(0) +
        f.when(col("rank_12").isNotNull(), weights["rank_12"]).otherwise(0)
    )
).orderBy('composite_rank', ascending=True)

combined

In [ ]:
#create csv
combined.write.mode("overwrite").option("header", True).csv("../data/curated/merchant_growth_rankings.csv")

In [ ]:
#create parquet
combined.write.parquet("../data/curated/growth_future_rev_rank", mode="overwrite")